# Reglas de asociación

# Dataset: Online Retail

## Contexto del Dataset

El conjunto de datos **Online Retail** contiene información histórica de ventas realizadas por una tienda minorista en línea durante el periodo comprendido entre **diciembre de 2010 y diciembre de 2011**.

Este dataset registra las transacciones comerciales realizadas por clientes de una tienda que vende productos principalmente de decoración y artículos para el hogar. Cada registro representa un producto vendido dentro de una factura, por lo que una misma compra puede estar formada por múltiples registros cuando un cliente adquiere diferentes artículos en una misma transacción.

El dataset está compuesto por **541,909 registros y 8 variables**, donde cada variable almacena información relacionada con la factura, producto, cantidad vendida, precio, cliente y ubicación geográfica de la compra.

Debido a que cada factura contiene los productos adquiridos por un cliente, este conjunto de datos permite analizar el comportamiento de compra y encontrar patrones relacionados con los artículos que suelen adquirirse juntos.

---

## Descripción de las variables del dataset

| Variable | Tipo de dato | Descripción |
|----------|--------------|-------------|
| **InvoiceNo** | Texto | Número de factura que identifica una transacción de compra. Permite agrupar los productos que pertenecen a una misma compra. |
| **StockCode** | Texto | Código único asignado a cada producto dentro del inventario de la tienda. |
| **Description** | Texto | Nombre o descripción del producto vendido. Contiene la información del artículo adquirido por el cliente. |
| **Quantity** | Entero | Cantidad de unidades compradas de un producto dentro de una factura. Los valores negativos representan devoluciones de productos. |
| **InvoiceDate** | Fecha/Hora | Fecha y hora en la que fue registrada la transacción de compra. |
| **UnitPrice** | Decimal | Precio unitario del producto vendido durante la transacción. |
| **CustomerID** | Decimal | Identificador único del cliente que realizó la compra. Permite relacionar las transacciones con cada comprador. |
| **Country** | Texto | País donde fue realizada la compra, permitiendo identificar la ubicación geográfica del cliente. |

---

## Información general del dataset

| Característica | Valor |
|----------------|-------|
| Número de registros | 541,909 |
| Número de variables | 8 |
| Periodo de datos | Diciembre 2010 - Diciembre 2011 |
| Tipo de información | Ventas y transacciones comerciales |
| Unidad principal de análisis | Facturas de compra |
| Variable principal para análisis de productos | Description |
| Identificador de transacción | InvoiceNo |

In [1]:
from pathlib import Path
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
DATA_PATH = Path('Online Retail.xlsx')
#COUNTRY = None
COUNTRY = 'United Kingdom'  # Cambiar por otro país; None utiliza todos.

Para el desarrollo es mejor dejar:

COUNTRY = 'United Kingdom'

No es recomendable usar None inicialmente porque, aunque el código seguirá funcionando, sí cambia:
- el consumo de memoria y tiempo;
- el número total de facturas;
- los valores de soporte;
- las reglas obtenidas;
- la interpretación comercial, al mezclar comportamientos de distintos países.

Además, con COUNTRY = None, el MIN_SUPPORT = 0.02 se calcula sobre todas las facturas. Una asociación fuerte en un país pequeño podría desaparecer al quedar diluida en el conjunto global.

In [2]:
data = pd.read_excel(DATA_PATH, sheet_name='Online Retail', engine='openpyxl')
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# Dimensiones del conjunto de datos
data.shape

(541909, 8)

In [4]:
# Limpieza necesaria antes de formar las transacciones
sales = data.dropna(subset=['InvoiceNo', 'Description']).copy()
sales['InvoiceNo'] = sales['InvoiceNo'].astype(str)
sales['Description'] = sales['Description'].astype(str).str.strip()
sales = sales[
    (sales['Quantity'] > 0)
    & (sales['UnitPrice'] > 0)
    & (~sales['InvoiceNo'].str.startswith('C'))
    & (sales['Description'] != '')
]
if COUNTRY is not None:
    sales = sales[sales['Country'].eq(COUNTRY)]

print(f'Líneas válidas: {len(sales):,}')
print(f'Facturas válidas: {sales["InvoiceNo"].nunique():,}')

Líneas válidas: 485,123
Facturas válidas: 18,019


In [5]:
# Lista de transacciones
# Se quitan artículos repetidos dentro de la misma factura.
transactions = (
    sales.groupby('InvoiceNo')['Description']
    .apply(lambda items: list(dict.fromkeys(items)))
    .tolist()
)

# Cantidad total de listas y visualización de las primeras 5.
print(f'Cantidad de listas de transacciones: {len(transactions):,}')
transactions[:5]

Cantidad de listas de transacciones: 18,019


[['WHITE HANGING HEART T-LIGHT HOLDER',
  'WHITE METAL LANTERN',
  'CREAM CUPID HEARTS COAT HANGER',
  'KNITTED UNION FLAG HOT WATER BOTTLE',
  'RED WOOLLY HOTTIE WHITE HEART.',
  'SET 7 BABUSHKA NESTING BOXES',
  'GLASS STAR FROSTED T-LIGHT HOLDER'],
 ['HAND WARMER UNION JACK', 'HAND WARMER RED POLKA DOT'],
 ['ASSORTED COLOUR BIRD ORNAMENT',
  "POPPY'S PLAYHOUSE BEDROOM",
  "POPPY'S PLAYHOUSE KITCHEN",
  'FELTCRAFT PRINCESS CHARLOTTE DOLL',
  'IVORY KNITTED MUG COSY',
  'BOX OF 6 ASSORTED COLOUR TEASPOONS',
  'BOX OF VINTAGE JIGSAW BLOCKS',
  'BOX OF VINTAGE ALPHABET BLOCKS',
  'HOME BUILDING BLOCK WORD',
  'LOVE BUILDING BLOCK WORD',
  'RECIPE BOX WITH METAL HEART',
  'DOORMAT NEW ENGLAND'],
 ['JAM MAKING SET WITH JARS',
  'RED COAT RACK PARIS FASHION',
  'YELLOW COAT RACK PARIS FASHION',
  'BLUE COAT RACK PARIS FASHION'],
 ['BATH BUILDING BLOCK WORD']]

In [6]:
# Diccionario de traducción para productos de las transacciones
traduccion_transacciones = {
    'WHITE HANGING HEART T-LIGHT HOLDER': 'Portavelas de corazón blanco colgante',
    'WHITE METAL LANTERN': 'Linterna metálica blanca',
    'CREAM CUPID HEARTS COAT HANGER': 'Percha de corazones Cupido color crema',
    'KNITTED UNION FLAG HOT WATER BOTTLE': 'Botella de agua caliente tejida con bandera de unión',
    'RED WOOLLY HOTTIE WHITE HEART.': 'Bolsa térmica roja de lana con corazón blanco',
    'SET 7 BABUSHKA NESTING BOXES': 'Juego de 7 cajas nido Babushka',
    'GLASS STAR FROSTED T-LIGHT HOLDER': 'Portavelas de estrella de vidrio esmerilado',

    'HAND WARMER UNION JACK': 'Calentador de manos con bandera de Reino Unido',
    'HAND WARMER RED POLKA DOT': 'Calentador de manos rojo con puntos',

    'ASSORTED COLOUR BIRD ORNAMENT': 'Adorno de pájaro de colores variados',
    "POPPY'S PLAYHOUSE BEDROOM": 'Habitación de casa de juegos de Poppy',
    "POPPY'S PLAYHOUSE KITCHEN": 'Cocina de casa de juegos de Poppy',
    'FELTCRAFT PRINCESS CHARLOTTE DOLL': 'Muñeca Princesa Charlotte de fieltro',
    'IVORY KNITTED MUG COSY': 'Funda tejida color marfil para taza',
    'BOX OF 6 ASSORTED COLOUR TEASPOONS': 'Caja de 6 cucharitas de colores variados',
    'BOX OF VINTAGE JIGSAW BLOCKS': 'Caja de bloques de rompecabezas vintage',
    'BOX OF VINTAGE ALPHABET BLOCKS': 'Caja de bloques de letras vintage',
    'HOME BUILDING BLOCK WORD': 'Bloques de construcción de palabras para hogar',
    'LOVE BUILDING BLOCK WORD': 'Bloques de construcción de palabras "Love"',
    'RECIPE BOX WITH METAL HEART': 'Caja de recetas con corazón metálico',
    'DOORMAT NEW ENGLAND': 'Tapete de entrada New England',

    'JAM MAKING SET WITH JARS': 'Juego para preparar mermelada con frascos',
    'RED COAT RACK PARIS FASHION': 'Perchero rojo estilo moda París',
    'YELLOW COAT RACK PARIS FASHION': 'Perchero amarillo estilo moda París',
    'BLUE COAT RACK PARIS FASHION': 'Perchero azul estilo moda París',

    'BATH BUILDING BLOCK WORD': 'Bloques de construcción de palabras para baño'
}

# Función para traducir productos de las transacciones
def traducir_transaccion(producto):
    return traduccion_transacciones.get(producto, producto)

# Crear copia de las transacciones en español
transactions_espanol = [
    [traducir_transaccion(producto) for producto in transaccion]
    for transaccion in transactions
]

# Mostrar primeras transacciones traducidas
transactions_espanol[:5]

[['Portavelas de corazón blanco colgante',
  'Linterna metálica blanca',
  'Percha de corazones Cupido color crema',
  'Botella de agua caliente tejida con bandera de unión',
  'Bolsa térmica roja de lana con corazón blanco',
  'Juego de 7 cajas nido Babushka',
  'Portavelas de estrella de vidrio esmerilado'],
 ['Calentador de manos con bandera de Reino Unido',
  'Calentador de manos rojo con puntos'],
 ['Adorno de pájaro de colores variados',
  'Habitación de casa de juegos de Poppy',
  'Cocina de casa de juegos de Poppy',
  'Muñeca Princesa Charlotte de fieltro',
  'Funda tejida color marfil para taza',
  'Caja de 6 cucharitas de colores variados',
  'Caja de bloques de rompecabezas vintage',
  'Caja de bloques de letras vintage',
  'Bloques de construcción de palabras para hogar',
  'Bloques de construcción de palabras "Love"',
  'Caja de recetas con corazón metálico',
  'Tapete de entrada New England'],
 ['Juego para preparar mermelada con frascos',
  'Perchero rojo estilo moda Par

In [7]:
# Crear un DataFrame con todas las transacciones
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

# Visualización de las primeras filas
print(f'Matriz: {df.shape[0]:,} facturas × {df.shape[1]:,} artículos')
df.head()

Matriz: 18,019 facturas × 3,996 artículos


,*Boombox Ipod Classic,*USB Office Mirror Ball,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Métricas de las reglas de asociación

- **Soporte:** proporción de facturas que contienen el conjunto.
- **Confianza:** probabilidad de comprar el consecuente cuando se compra el antecedente.
- **Lift:** fuerza frente a compras independientes. Un valor mayor que 1 indica asociación positiva.

Para cada métrica se muestran las 20 mejores reglas, ordenadas de mayor a menor. Primero aparece la tabla con los nombres originales en inglés, después la misma tabla traducida al español y, finalmente, la interpretación de sus primeras 5 reglas.

In [8]:
# Generar conjuntos frecuentes y reglas de asociación.
frequent_itemsets = apriori(
    df, min_support=0.02, use_colnames=True, max_len=3, low_memory=True
)
rules = association_rules(
    frequent_itemsets, metric='lift', min_threshold=1.20, num_itemsets=len(df)
)

print(f'Conjuntos frecuentes: {len(frequent_itemsets):,}')
print(f'Reglas encontradas: {len(rules):,}')

Conjuntos frecuentes: 401
Reglas encontradas: 218


In [9]:
# Traducciones de los productos que aparecen en las mejores reglas.
traduccion_productos = {
    'ALARM CLOCK BAKELIKE GREEN': 'Reloj despertador estilo bakelite verde',
    'ALARM CLOCK BAKELIKE RED': 'Reloj despertador estilo bakelite rojo',
    'CHARLOTTE BAG PINK POLKADOT': 'Bolsa Charlotte rosa con lunares',
    'DOLLY GIRL LUNCH BOX': 'Lonchera Dolly Girl',
    'DOTCOM POSTAGE': 'Gastos de envío Dotcom',
    'GARDENERS KNEELING PAD CUP OF TEA': 'Almohadilla de jardinería con taza de té',
    'GARDENERS KNEELING PAD KEEP CALM': 'Almohadilla de jardinería Keep Calm',
    'GREEN REGENCY TEACUP AND SAUCER': 'Taza y plato de té Regency verde',
    'JUMBO  BAG BAROQUE BLACK WHITE': 'Bolsa jumbo barroca negra y blanca',
    'JUMBO BAG APPLES': 'Bolsa jumbo con manzanas',
    'JUMBO BAG PEARS': 'Bolsa jumbo con peras',
    'JUMBO BAG PINK POLKADOT': 'Bolsa jumbo rosa con lunares',
    'JUMBO BAG RED RETROSPOT': 'Bolsa jumbo roja Retrospot',
    'JUMBO BAG SCANDINAVIAN BLUE PAISLEY': 'Bolsa jumbo azul escandinava con diseño paisley',
    'JUMBO SHOPPER VINTAGE RED PAISLEY': 'Bolsa grande vintage roja con diseño paisley',
    'JUMBO STORAGE BAG SUKI': 'Bolsa jumbo de almacenamiento Suki',
    'LUNCH BAG  BLACK SKULL.': 'Lonchera negra con calaveras',
    'LUNCH BAG PINK POLKADOT': 'Lonchera rosa con lunares',
    'LUNCH BAG RED RETROSPOT': 'Lonchera roja Retrospot',
    'PINK REGENCY TEACUP AND SAUCER': 'Taza y plato de té Regency rosa',
    'RED RETROSPOT CHARLOTTE BAG': 'Bolsa Charlotte roja Retrospot',
    'ROSES REGENCY TEACUP AND SAUCER': 'Taza y plato de té Regency con rosas',
    'SPACEBOY LUNCH BOX': 'Lonchera Spaceboy',
    'STRAWBERRY CHARLOTTE BAG': 'Bolsa Charlotte con fresas',
    'WOODEN HEART CHRISTMAS SCANDINAVIAN': 'Corazón navideño de madera escandinavo',
    'WOODEN STAR CHRISTMAS SCANDINAVIAN': 'Estrella navideña de madera escandinava',
    'WOODLAND CHARLOTTE BAG': 'Bolsa Charlotte con diseño de bosque'
}

def itemset_a_texto(itemset, traducir=False):
    productos = sorted(itemset)
    if traducir:
        productos = [traduccion_productos.get(p, p) for p in productos]
    return ', '.join(productos)

In [10]:
from IPython.display import display

TOP_REGLAS = 20

def ordenar_reglas(metrica, cantidad=TOP_REGLAS):
    otras = [m for m in ['support', 'confidence', 'lift'] if m != metrica]
    return (
        rules.sort_values([metrica] + otras, ascending=False)
        .head(cantidad)
        .reset_index(drop=True)
    )

def crear_tabla(reglas_ordenadas, metrica, traducir=False):
    tabla = pd.DataFrame({
        'antecedent': reglas_ordenadas['antecedents'].apply(
            lambda x: itemset_a_texto(x, traducir)
        ),
        'consequent': reglas_ordenadas['consequents'].apply(
            lambda x: itemset_a_texto(x, traducir)
        ),
        metrica: reglas_ordenadas[metrica]
    })

    if traducir:
        nombres = {
            'antecedent': 'antecedente',
            'consequent': 'consecuente',
            'support': 'soporte',
            'confidence': 'confianza',
            'lift': 'lift'
        }
        tabla = tabla.rename(columns=nombres)

    return tabla.round(3)

def mostrar_tablas(reglas_ordenadas, metrica):
    print('Table in English')
    display(crear_tabla(reglas_ordenadas, metrica))
    print('Tabla en español')
    display(crear_tabla(reglas_ordenadas, metrica, traducir=True))

In [11]:
def elegir_uso(regla, enfoque):
    # El criterio cambia según la métrica que se está priorizando.
    if enfoque == 'support':
        if regla['support'] >= 0.04:
            return 'promoción'
        if regla['lift'] >= 10:
            return 'paquete'
        return 'recomendación'

    if enfoque == 'confidence':
        if regla['support'] >= 0.03:
            return 'promoción'
        if regla['confidence'] >= 0.88:
            return 'recomendación'
        if regla['lift'] >= 15:
            return 'paquete'
        return 'recomendación'

    if enfoque == 'lift':
        if regla['confidence'] < 0.60:
            return 'promoción'
        if regla['confidence'] >= 0.80:
            return 'recomendación'
        return 'paquete'

    if regla['support'] >= 0.035:
        return 'promoción'
    if regla['confidence'] >= 0.80:
        return 'recomendación'
    return 'paquete'

In [12]:
def interpretar_regla(regla, numero, enfoque):
    x = itemset_a_texto(regla['antecedents'], traducir=True)
    y = itemset_a_texto(regla['consequents'], traducir=True)

    if enfoque == 'support':
        interpretacion = (
            f'esta asociación aparece en {regla["support"]:.1%} de las facturas. '
            f'Entre quienes compran {x}, {regla["confidence"]:.1%} también compra {y}; '
            f'el lift de {regla["lift"]:.3f} confirma una relación positiva.'
        )
    elif enfoque == 'confidence':
        interpretacion = (
            f'los clientes que compran {x} también compran {y} en '
            f'{regla["confidence"]:.1%} de los casos. La regla cubre '
            f'{regla["support"]:.1%} de las facturas y la relación es positiva.'
        )
    elif enfoque == 'lift':
        interpretacion = (
            f'la compra de {x} y {y} ocurre {regla["lift"]:.3f} veces más de lo '
            f'esperado si fueran compras independientes. La confianza es '
            f'{regla["confidence"]:.1%} y alcanza {regla["support"]:.1%} de las facturas.'
        )
    else:
        interpretacion = (
            f'la regla combina un soporte de {regla["support"]:.1%}, una confianza de '
            f'{regla["confidence"]:.1%} y un lift de {regla["lift"]:.3f}; por ello '
            'mantiene un equilibrio favorable entre alcance, probabilidad y fuerza.'
        )

    uso = elegir_uso(regla, enfoque)
    usos = {
        'recomendación': f'mostrar {y} como recomendación al comprar {x}, porque la probabilidad de respuesta es alta',
        'paquete': f'crear un paquete con {x} y {y}, porque son artículos fuertemente complementarios',
        'promoción': f'ofrecer una promoción cruzada de {y} al comprar {x}, para aprovechar su alcance o impulsar la conversión'
    }

    print(f'REGLA {numero}: {{{x}}} → {{{y}}}')
    print(f'Soporte: {regla["support"]:.3f} ({regla["support"]:.1%} de las facturas)')
    print(f'Confianza: {regla["confidence"]:.3f} ({regla["confidence"]:.1%})')
    print(f'Lift: {regla["lift"]:.3f}')
    print(f'Interpretación: {interpretacion}')
    print(f'Uso posible ({uso}): {usos[uso]}.')
    print('-' * 100)

def interpretar_primeras_cinco(reglas_ordenadas, enfoque):
    print('Interpretación de las primeras 5 reglas')
    for numero, (_, regla) in enumerate(reglas_ordenadas.head(5).iterrows(), start=1):
        interpretar_regla(regla, numero, enfoque)

In [13]:
print('MÉTRICA: SOPORTE')
top_soporte = ordenar_reglas('support')
mostrar_tablas(top_soporte, 'support')
interpretar_primeras_cinco(top_soporte, 'support')

MÉTRICA: SOPORTE
Table in English


,antecedent,consequent,support
0,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,0.044
1,JUMBO BAG RED RETROSPOT,JUMBO BAG PINK POLKADOT,0.044
2,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.039
3,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.039
4,JUMBO STORAGE BAG SUKI,JUMBO BAG RED RETROSPOT,0.039
5,JUMBO BAG RED RETROSPOT,JUMBO STORAGE BAG SUKI,0.039
6,JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO BAG RED RETROSPOT,0.036
7,JUMBO BAG RED RETROSPOT,JUMBO SHOPPER VINTAGE RED PAISLEY,0.036
8,LUNCH BAG BLACK SKULL.,LUNCH BAG RED RETROSPOT,0.034
9,LUNCH BAG RED RETROSPOT,LUNCH BAG BLACK SKULL.,0.034


Tabla en español


,antecedente,consecuente,soporte
0,Bolsa jumbo rosa con lunares,Bolsa jumbo roja Retrospot,0.044
1,Bolsa jumbo roja Retrospot,Bolsa jumbo rosa con lunares,0.044
2,Taza y plato de té Regency verde,Taza y plato de té Regency con rosas,0.039
3,Taza y plato de té Regency con rosas,Taza y plato de té Regency verde,0.039
4,Bolsa jumbo de almacenamiento Suki,Bolsa jumbo roja Retrospot,0.039
5,Bolsa jumbo roja Retrospot,Bolsa jumbo de almacenamiento Suki,0.039
6,Bolsa grande vintage roja con diseño paisley,Bolsa jumbo roja Retrospot,0.036
7,Bolsa jumbo roja Retrospot,Bolsa grande vintage roja con diseño paisley,0.036
8,Lonchera negra con calaveras,Lonchera roja Retrospot,0.034
9,Lonchera roja Retrospot,Lonchera negra con calaveras,0.034


Interpretación de las primeras 5 reglas
REGLA 1: {Bolsa jumbo rosa con lunares} → {Bolsa jumbo roja Retrospot}
Soporte: 0.044 (4.4% de las facturas)
Confianza: 0.677 (67.7%)
Lift: 6.307
Interpretación: esta asociación aparece en 4.4% de las facturas. Entre quienes compran Bolsa jumbo rosa con lunares, 67.7% también compra Bolsa jumbo roja Retrospot; el lift de 6.307 confirma una relación positiva.
Uso posible (promoción): ofrecer una promoción cruzada de Bolsa jumbo roja Retrospot al comprar Bolsa jumbo rosa con lunares, para aprovechar su alcance o impulsar la conversión.
----------------------------------------------------------------------------------------------------
REGLA 2: {Bolsa jumbo roja Retrospot} → {Bolsa jumbo rosa con lunares}
Soporte: 0.044 (4.4% de las facturas)
Confianza: 0.406 (40.6%)
Lift: 6.307
Interpretación: esta asociación aparece en 4.4% de las facturas. Entre quienes compran Bolsa jumbo roja Retrospot, 40.6% también compra Bolsa jumbo rosa con lunares; el lift

In [14]:
print('MÉTRICA: CONFIANZA')
top_confianza = ordenar_reglas('confidence')
mostrar_tablas(top_confianza, 'confidence')
interpretar_primeras_cinco(top_confianza, 'confidence')

MÉTRICA: CONFIANZA
Table in English


,antecedent,consequent,confidence
0,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,0.903
1,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.854
2,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.821
3,"JUMBO BAG PINK POLKADOT, JUMBO SHOPPER VINTAGE...",JUMBO BAG RED RETROSPOT,0.806
4,"JUMBO BAG PINK POLKADOT, JUMBO STORAGE BAG SUKI",JUMBO BAG RED RETROSPOT,0.802
5,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.776
6,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.768
7,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.751
8,"JUMBO SHOPPER VINTAGE RED PAISLEY, JUMBO STORA...",JUMBO BAG RED RETROSPOT,0.749
9,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.732


Tabla en español


,antecedente,consecuente,confianza
0,"Taza y plato de té Regency rosa, Taza y plato ...",Taza y plato de té Regency verde,0.903
1,"Taza y plato de té Regency verde, Taza y plato...",Taza y plato de té Regency con rosas,0.854
2,Taza y plato de té Regency rosa,Taza y plato de té Regency verde,0.821
3,"Bolsa jumbo rosa con lunares, Bolsa grande vin...",Bolsa jumbo roja Retrospot,0.806
4,"Bolsa jumbo rosa con lunares, Bolsa jumbo de a...",Bolsa jumbo roja Retrospot,0.802
5,Taza y plato de té Regency rosa,Taza y plato de té Regency con rosas,0.776
6,Estrella navideña de madera escandinava,Corazón navideño de madera escandinavo,0.768
7,Taza y plato de té Regency verde,Taza y plato de té Regency con rosas,0.751
8,"Bolsa grande vintage roja con diseño paisley, ...",Bolsa jumbo roja Retrospot,0.749
9,Taza y plato de té Regency con rosas,Taza y plato de té Regency verde,0.732


Interpretación de las primeras 5 reglas
REGLA 1: {Taza y plato de té Regency rosa, Taza y plato de té Regency con rosas} → {Taza y plato de té Regency verde}
Soporte: 0.027 (2.7% de las facturas)
Confianza: 0.903 (90.3%)
Lift: 17.454
Interpretación: los clientes que compran Taza y plato de té Regency rosa, Taza y plato de té Regency con rosas también compran Taza y plato de té Regency verde en 90.3% de los casos. La regla cubre 2.7% de las facturas y la relación es positiva.
Uso posible (recomendación): mostrar Taza y plato de té Regency verde como recomendación al comprar Taza y plato de té Regency rosa, Taza y plato de té Regency con rosas, porque la probabilidad de respuesta es alta.
----------------------------------------------------------------------------------------------------
REGLA 2: {Taza y plato de té Regency verde, Taza y plato de té Regency rosa} → {Taza y plato de té Regency con rosas}
Soporte: 0.027 (2.7% de las facturas)
Confianza: 0.854 (85.4%)
Lift: 16.100
Interpret

### Lift

Las reglas se ordenan por lift de mayor a menor. La interpretación considera sus primeras 5 posiciones.

In [15]:
print('MÉTRICA: LIFT')
top_lift = ordenar_reglas('lift')
mostrar_tablas(top_lift, 'lift')
interpretar_primeras_cinco(top_lift, 'lift')

MÉTRICA: LIFT
Table in English


,antecedent,consequent,lift
0,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,27.197
1,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,27.197
2,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER,18.041
3,PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",18.041
4,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,17.454
5,GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",17.454
6,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,16.100
7,ROSES REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...",16.100
8,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,15.864
9,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,15.864


Tabla en español


,antecedente,consecuente,lift
0,Corazón navideño de madera escandinavo,Estrella navideña de madera escandinava,27.197
1,Estrella navideña de madera escandinava,Corazón navideño de madera escandinavo,27.197
2,"Taza y plato de té Regency verde, Taza y plato...",Taza y plato de té Regency rosa,18.041
3,Taza y plato de té Regency rosa,"Taza y plato de té Regency verde, Taza y plato...",18.041
4,"Taza y plato de té Regency rosa, Taza y plato ...",Taza y plato de té Regency verde,17.454
5,Taza y plato de té Regency verde,"Taza y plato de té Regency rosa, Taza y plato ...",17.454
6,"Taza y plato de té Regency verde, Taza y plato...",Taza y plato de té Regency con rosas,16.100
7,Taza y plato de té Regency con rosas,"Taza y plato de té Regency verde, Taza y plato...",16.100
8,Taza y plato de té Regency verde,Taza y plato de té Regency rosa,15.864
9,Taza y plato de té Regency rosa,Taza y plato de té Regency verde,15.864


Interpretación de las primeras 5 reglas
REGLA 1: {Corazón navideño de madera escandinavo} → {Estrella navideña de madera escandinava}
Soporte: 0.020 (2.0% de las facturas)
Confianza: 0.723 (72.3%)
Lift: 27.197
Interpretación: la compra de Corazón navideño de madera escandinavo y Estrella navideña de madera escandinava ocurre 27.197 veces más de lo esperado si fueran compras independientes. La confianza es 72.3% y alcanza 2.0% de las facturas.
Uso posible (paquete): crear un paquete con Corazón navideño de madera escandinavo y Estrella navideña de madera escandinava, porque son artículos fuertemente complementarios.
----------------------------------------------------------------------------------------------------
REGLA 2: {Estrella navideña de madera escandinava} → {Corazón navideño de madera escandinavo}
Soporte: 0.020 (2.0% de las facturas)
Confianza: 0.768 (76.8%)
Lift: 27.197
Interpretación: la compra de Estrella navideña de madera escandinava y Corazón navideño de madera escandin

In [16]:
# Puntaje general: promedio de soporte, confianza y lift normalizados.
reglas_generales = rules.copy()
for metrica in ['support', 'confidence', 'lift']:
    reglas_generales[f'{metrica}_normalizado'] = (
        reglas_generales[metrica] / reglas_generales[metrica].max()
    )

reglas_generales['puntaje_general'] = reglas_generales[[
    'support_normalizado', 'confidence_normalizado', 'lift_normalizado'
]].mean(axis=1)

top_general = (
    reglas_generales.sort_values('puntaje_general', ascending=False)
    .head(TOP_REGLAS)
    .reset_index(drop=True)
)

## Mejores reglas de manera general

Para combinar las tres métricas sin favorecer sus distintas escalas, cada una se divide entre su valor máximo y después se calcula el promedio. La tabla se ordena por ese puntaje general, pero muestra únicamente los valores originales de soporte, confianza y lift.

In [17]:
def crear_tabla_general(reglas_ordenadas, traducir=False):
    tabla = pd.DataFrame({
        'antecedent': reglas_ordenadas['antecedents'].apply(
            lambda x: itemset_a_texto(x, traducir)
        ),
        'consequent': reglas_ordenadas['consequents'].apply(
            lambda x: itemset_a_texto(x, traducir)
        ),
        'support': reglas_ordenadas['support'],
        'confidence': reglas_ordenadas['confidence'],
        'lift': reglas_ordenadas['lift']
    })

    if traducir:
        tabla = tabla.rename(columns={
            'antecedent': 'antecedente',
            'consequent': 'consecuente',
            'support': 'soporte',
            'confidence': 'confianza'
        })
    return tabla.round(3)

print('MEJORES REGLAS SEGÚN LAS TRES MÉTRICAS')
print('Table in English')
display(crear_tabla_general(top_general))
print('Tabla en español')
display(crear_tabla_general(top_general, traducir=True))
interpretar_primeras_cinco(top_general, 'general')

MEJORES REGLAS SEGÚN LAS TRES MÉTRICAS
Table in English


,antecedent,consequent,support,confidence,lift
0,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.020,0.768,27.197
1,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.020,0.723,27.197
2,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,0.027,0.903,17.454
3,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.039,0.751,14.156
4,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.032,0.821,15.864
5,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.039,0.732,14.156
6,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.027,0.854,16.100
7,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.030,0.776,14.633
8,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER,0.027,0.703,18.041
9,PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",0.027,0.701,18.041


Tabla en español


,antecedente,consecuente,soporte,confianza,lift
0,Estrella navideña de madera escandinava,Corazón navideño de madera escandinavo,0.020,0.768,27.197
1,Corazón navideño de madera escandinavo,Estrella navideña de madera escandinava,0.020,0.723,27.197
2,"Taza y plato de té Regency rosa, Taza y plato ...",Taza y plato de té Regency verde,0.027,0.903,17.454
3,Taza y plato de té Regency verde,Taza y plato de té Regency con rosas,0.039,0.751,14.156
4,Taza y plato de té Regency rosa,Taza y plato de té Regency verde,0.032,0.821,15.864
5,Taza y plato de té Regency con rosas,Taza y plato de té Regency verde,0.039,0.732,14.156
6,"Taza y plato de té Regency verde, Taza y plato...",Taza y plato de té Regency con rosas,0.027,0.854,16.100
7,Taza y plato de té Regency rosa,Taza y plato de té Regency con rosas,0.030,0.776,14.633
8,"Taza y plato de té Regency verde, Taza y plato...",Taza y plato de té Regency rosa,0.027,0.703,18.041
9,Taza y plato de té Regency rosa,"Taza y plato de té Regency verde, Taza y plato...",0.027,0.701,18.041


Interpretación de las primeras 5 reglas
REGLA 1: {Estrella navideña de madera escandinava} → {Corazón navideño de madera escandinavo}
Soporte: 0.020 (2.0% de las facturas)
Confianza: 0.768 (76.8%)
Lift: 27.197
Interpretación: la regla combina un soporte de 2.0%, una confianza de 76.8% y un lift de 27.197; por ello mantiene un equilibrio favorable entre alcance, probabilidad y fuerza.
Uso posible (paquete): crear un paquete con Estrella navideña de madera escandinava y Corazón navideño de madera escandinavo, porque son artículos fuertemente complementarios.
----------------------------------------------------------------------------------------------------
REGLA 2: {Corazón navideño de madera escandinavo} → {Estrella navideña de madera escandinava}
Soporte: 0.020 (2.0% de las facturas)
Confianza: 0.723 (72.3%)
Lift: 27.197
Interpretación: la regla combina un soporte de 2.0%, una confianza de 72.3% y un lift de 27.197; por ello mantiene un equilibrio favorable entre alcance, probabilidad